In [62]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [63]:
df = pd.read_csv('winequality-total.csv', sep=';')

X = df.drop('quality', axis=1).values
y_raw = df['quality'].values

y = np.zeros((len(y_raw), 10))
for i in range(len(y_raw)):
    if y_raw[i] < 10:
        y[i, int(y_raw[i])] = 1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [64]:
def ReLU(X):
  return np.maximum(X, 0)

In [65]:
def ReLUDerivative(x):
  return (x > 0).astype(float)

In [66]:
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

In [67]:
alpha = 0.01
hidden_size_1 = 4
hidden_size_2 = 3
epoch = 100
output = 10

np.random.seed(42)
weight_0_1 = np.random.random((X_train_scaled.shape[1], hidden_size_1)) * 0.1
weight_1_2 = np.random.random((hidden_size_1, hidden_size_2)) * 0.1
weight_2_3 = np.random.random((hidden_size_2, output)) * 0.1

In [68]:
for i in range(epoch):
  layer_3_error = 0
  for j in range(len(X_train_scaled)):
    layer_0 = X_train_scaled[j:j+1]
    target = y_train[j:j+1]
    layer_1 = ReLU(np.dot(layer_0, weight_0_1))
    layer_2 = ReLU(np.dot(layer_1, weight_1_2))
    logits = np.dot(layer_2, weight_2_3)
    layer_3 = softmax(logits)
    delta_layer_3 = layer_3 - target
    layer_3_error += np.sum(delta_layer_3**2)
    delta_layer_2 = np.dot(delta_layer_3, weight_2_3.T) * ReLUDerivative(layer_2)
    delta_layer_1 = np.dot(delta_layer_2, weight_1_2.T) * ReLUDerivative(layer_1)
    weight_2_3 -= alpha * np.dot(layer_2.T, delta_layer_3)
    weight_1_2 -= alpha * np.dot(layer_1.T, delta_layer_2)
    weight_0_1 -= alpha * np.dot(layer_0.T, delta_layer_1)
  if i % 10 == 0:
    print(f"Epoch {i} Loss: {layer_3_error:.4f}")

test_pred_logits = np.dot(ReLU(np.dot(ReLU(np.dot(X_test_scaled, weight_0_1)), weight_1_2)), weight_2_3)
test_pred = softmax(test_pred_logits)

accuracy = np.mean(np.argmax(test_pred, axis=1) == np.argmax(y_test, axis=1))
print(f"Final Accuracy: {accuracy * 100:.2f}%")

Epoch 0 Loss: 4066.1631
Epoch 10 Loss: 3188.5044
Epoch 20 Loss: 3157.5713
Epoch 30 Loss: 3154.0290
Epoch 40 Loss: 3147.7384
Epoch 50 Loss: 3142.9480
Epoch 60 Loss: 3132.3692
Epoch 70 Loss: 3125.3921
Epoch 80 Loss: 3123.4184
Epoch 90 Loss: 3124.7348
Final Accuracy: 53.31%
